In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

In [0]:
bronze_table = "dev.01_bronze.sales_transactions"
silver_table = "dev.02_silver.sales_transactions"

In [0]:
df = spark.table(bronze_table)

In [0]:
df.printSchema()
display(df.limit(10))
df.count()

In [0]:
df_clean = df.dropna(subset=["transactionID"]).fillna("unknown", subset=["customerID"])

In [0]:
df_clean = (

    df_clean
    .withColumn("customerID", F.col("customerID").cast("integer") )
    .withColumn("paymentMethod", F.lower(F.col("paymentMethod")) )
    )

In [0]:
df_clean = df_clean.dropDuplicates(["transactionID"])

In [0]:
df_clean.printSchema()
display(df_clean.limit(10))

In [0]:
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)